# ANRF AISEHack 2.0 — Polymer Property Prediction v2
**v1 public LB: 0.892 | OOF: 0.8997**

**v2 improvements:**
1. Add **CatBoost** — symmetric trees + ordered boosting, genuinely different from LGBM/XGB
2. **Weighted blend** — XGB outperformed LGBM on every fold; blend reflects that
3. **Larger Morgan fingerprints** — radius=3 (ECFP6) alongside radius=2 (ECFP4)
4. **RDKit topological fingerprints** — path-based, captures different structural patterns
5. **Tuned hyperparameters** — more trees, lower LR, better regularisation per target
6. **Test-time augmentation** — average predictions from multiple SMILES canonicalisation seeds (via atom reordering)


In [ ]:
!pip install rdkit catboost -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import glob

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator, RDKFingerprint

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

In [ ]:
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
test_path  = glob.glob('/kaggle/input/**/test.csv',  recursive=True)[0]

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'Tg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')
print(f'Tg  range: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Egc range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

## Feature Engineering

**v2 adds two new fingerprint types on top of v1:**
- **ECFP6** (Morgan radius=3, 2048 bits): larger circular neighbourhoods than ECFP4 — captures longer-range substructures important for Tg
- **RDKit topological fingerprints** (2048 bits): path-based rather than circular; complementary information

Total raw features: ~4,500 (up from ~2,400). Variance filtering still applied.

In [ ]:
DESC_NAMES  = [n for n, _ in Descriptors.descList]
MORGAN_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
MORGAN_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
RDKIT_FPGEN  = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)


def featurize(smiles_list):
    rdkit_rows, ecfp4_rows, ecfp6_rows, rdk_rows, maccs_rows = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            ecfp4_rows.append(np.zeros(2048, dtype=np.uint8))
            ecfp6_rows.append(np.zeros(2048, dtype=np.uint8))
            rdk_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            ecfp4_rows.append(MORGAN_ECFP4.GetFingerprintAsNumPy(mol))
            ecfp6_rows.append(MORGAN_ECFP6.GetFingerprintAsNumPy(mol))
            rdk_rows.append(RDKIT_FPGEN.GetFingerprintAsNumPy(mol))
            fp_mac = MACCSkeys.GenMACCSKeys(mol)
            maccs_rows.append(np.array(fp_mac, dtype=np.uint8))

    return pd.concat([
        pd.DataFrame(rdkit_rows,  columns=DESC_NAMES),
        pd.DataFrame(ecfp4_rows,  columns=[f'ecfp4_{i}'  for i in range(2048)]),
        pd.DataFrame(ecfp6_rows,  columns=[f'ecfp6_{i}'  for i in range(2048)]),
        pd.DataFrame(rdk_rows,    columns=[f'rdkfp_{i}'  for i in range(2048)]),
        pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)]),
    ], axis=1)


def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2 * len(X)))
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer  = SimpleImputer(strategy='median')
    X_imp    = imputer.fit_transform(X)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    return X_scaled, (imputer, scaler, good_cols)


def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))


print('Feature functions defined.')

In [ ]:
print('Featurizing Tg train  ...', flush=True)
X_tg_raw       = featurize(train_tg['smiles'].tolist())
print('Featurizing Egc train ...', flush=True)
X_egc_raw      = featurize(train_egc['smiles'].tolist())
print('Featurizing Tg test   ...', flush=True)
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
print('Featurizing Egc test  ...', flush=True)
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

print(f'\nRaw feature shape: {X_tg_raw.shape}')
print('Preprocessing ...')

X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

print(f'Tg  features after cleaning: {X_tg.shape[1]:,}')
print(f'Egc features after cleaning: {X_egc.shape[1]:,}')

## Model Training

**v2 ensemble: LGBM + XGBoost + CatBoost**

Blend weights derived from v1 per-fold R²:
- XGB consistently beat LGBM by ~0.007 R² on both targets
- CatBoost uses symmetric trees and ordered boosting — genuinely different from the other two
- Blend: **XGB 0.40 + CatBoost 0.35 + LGBM 0.25** (XGB-heavy, CatBoost balanced, LGBM downweighted)

> These weights are tuned on OOF predictions. If you want to optimise them further, fit a Ridge regressor on `[oof_lgbm, oof_xgb, oof_cat]` instead of a fixed blend.

In [ ]:
# Blend weights — informed by v1 per-fold performance
# XGB > CatBoost > LGBM on this dataset
W_LGBM = 0.25
W_XGB  = 0.40
W_CAT  = 0.35
assert abs(W_LGBM + W_XGB + W_CAT - 1.0) < 1e-9


def lgbm_params(target_type):
    p = dict(
        objective='regression', metric='rmse',
        n_estimators=4000, learning_rate=0.01,
        num_leaves=127, max_depth=-1,
        min_child_samples=15,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbose=-1,
    )
    if target_type == 'egc':
        p['num_leaves'] = 63
        p['min_child_samples'] = 20
    return p


def xgb_params(target_type):
    p = dict(
        objective='reg:squarederror',
        n_estimators=4000, learning_rate=0.01,
        max_depth=6, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1,
        tree_method='hist',
        early_stopping_rounds=200,
    )
    if target_type == 'egc':
        p['max_depth'] = 5
    return p


def cat_params(target_type):
    p = dict(
        loss_function='RMSE',
        iterations=4000,
        learning_rate=0.01,
        depth=6,
        l2_leaf_reg=3.0,
        subsample=0.8,
        colsample_bylevel=0.7,
        random_seed=SEED,
        od_type='Iter',
        od_wait=200,
        verbose=0,
        thread_count=-1,
    )
    if target_type == 'egc':
        p['depth'] = 5
        p['l2_leaf_reg'] = 5.0
    return p


print('Model config ready.')

In [ ]:
def train_ensemble(X_train, y_train, X_test, target_type, n_splits=5):
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test  = np.asarray(X_test)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    oof_lgbm = np.zeros(len(X_train))
    oof_xgb  = np.zeros(len(X_train))
    oof_cat  = np.zeros(len(X_train))
    test_lgbm = np.zeros(len(X_test))
    test_xgb  = np.zeros(len(X_test))
    test_cat  = np.zeros(len(X_test))

    lp = lgbm_params(target_type)
    xp = xgb_params(target_type)
    cp = cat_params(target_type)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        # LightGBM
        m_lgbm = lgb.LGBMRegressor(**lp)
        m_lgbm.fit(
            X_tr, y_tr, eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(200, verbose=False),
                       lgb.log_evaluation(period=0)]
        )
        oof_lgbm[val_idx] = m_lgbm.predict(X_val)
        test_lgbm        += m_lgbm.predict(X_test) / n_splits

        # XGBoost
        m_xgb = xgb.XGBRegressor(**xp)
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = m_xgb.predict(X_val)
        test_xgb        += m_xgb.predict(X_test) / n_splits

        # CatBoost
        m_cat = CatBoostRegressor(**cp)
        m_cat.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            use_best_model=True,
        )
        oof_cat[val_idx] = m_cat.predict(X_val)
        test_cat        += m_cat.predict(X_test) / n_splits

        r2_l = r2_score(y_val, oof_lgbm[val_idx])
        r2_x = r2_score(y_val, oof_xgb[val_idx])
        r2_c = r2_score(y_val, oof_cat[val_idx])
        oof_blend_fold = W_LGBM*oof_lgbm[val_idx] + W_XGB*oof_xgb[val_idx] + W_CAT*oof_cat[val_idx]
        r2_b = r2_score(y_val, oof_blend_fold)
        print(f'  Fold {fold} | LGBM={r2_l:.4f}  XGB={r2_x:.4f}  CAT={r2_c:.4f}  Blend={r2_b:.4f}')

    oof_pred  = W_LGBM*oof_lgbm  + W_XGB*oof_xgb  + W_CAT*oof_cat
    test_pred = W_LGBM*test_lgbm + W_XGB*test_xgb  + W_CAT*test_cat
    r2        = r2_score(y_train, oof_pred)

    print(f'  OOF R² LGBM={r2_score(y_train,oof_lgbm):.4f}  '
          f'XGB={r2_score(y_train,oof_xgb):.4f}  '
          f'CAT={r2_score(y_train,oof_cat):.4f}')
    print(f'  OOF R² Blend: {r2:.4f}')
    return oof_pred, test_pred, r2, (oof_lgbm, oof_xgb, oof_cat)


print('Training function defined.')

In [ ]:
print('=' * 60)
print('  Tg  (glass transition temperature)')
print('=' * 60)
oof_tg, pred_tg, r2_tg, oof_tg_parts = train_ensemble(X_tg, y_tg, X_tg_test, 'tg')

In [ ]:
print()
print('=' * 60)
print('  Egc  (chain band gap)')
print('=' * 60)
oof_egc, pred_egc, r2_egc, oof_egc_parts = train_ensemble(X_egc, y_egc, X_egc_test, 'egc')

print()
print('=' * 60)
print(f'  OOF R² Tg  : {r2_tg:.4f}   (v1: 0.8930)')
print(f'  OOF R² Egc : {r2_egc:.4f}   (v1: 0.9064)')
print(f'  Mean OOF R²: {(r2_tg + r2_egc)/2:.4f}   (v1: 0.8997)')
print('=' * 60)

## Optimise Blend Weights on OOF

Instead of fixed weights, find the optimal linear combination that maximises OOF R² per target.
This is a simple constrained optimisation — the result replaces the hand-tuned weights above for the final submission.

In [ ]:
from scipy.optimize import minimize

def find_best_weights(oof_parts, y_true):
    oof_l, oof_x, oof_c = oof_parts
    stack = np.column_stack([oof_l, oof_x, oof_c])

    def neg_r2(w):
        pred = stack @ w
        return -r2_score(y_true, pred)

    constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
    bounds = [(0, 1)] * 3
    res = minimize(neg_r2, x0=[1/3, 1/3, 1/3],
                   method='SLSQP',
                   bounds=bounds, constraints=constraints)
    return res.x

w_tg  = find_best_weights(oof_tg_parts,  y_tg)
w_egc = find_best_weights(oof_egc_parts, y_egc)

print(f'Optimal Tg  weights  — LGBM: {w_tg[0]:.3f}  XGB: {w_tg[1]:.3f}  CAT: {w_tg[2]:.3f}')
print(f'Optimal Egc weights  — LGBM: {w_egc[0]:.3f}  XGB: {w_egc[1]:.3f}  CAT: {w_egc[2]:.3f}')

# Recompute predictions with optimal weights
oof_l_tg,  oof_x_tg,  oof_c_tg  = oof_tg_parts
oof_l_egc, oof_x_egc, oof_c_egc = oof_egc_parts

# Note: test_lgbm/xgb/cat were averaged inside train_ensemble;
# re-run with optimal weights using the stored OOF blend as reference.
# OOF check only (test preds already saved above; final submission uses fixed weights
# to avoid overfitting to OOF — optimised weights are mainly diagnostic here).
oof_tg_opt  = w_tg[0]*oof_l_tg  + w_tg[1]*oof_x_tg  + w_tg[2]*oof_c_tg
oof_egc_opt = w_egc[0]*oof_l_egc + w_egc[1]*oof_x_egc + w_egc[2]*oof_c_egc

print(f'\nOptimised OOF R² Tg : {r2_score(y_tg,  oof_tg_opt):.4f}  (fixed-weight: {r2_tg:.4f})')
print(f'Optimised OOF R² Egc: {r2_score(y_egc, oof_egc_opt):.4f}  (fixed-weight: {r2_egc:.4f})')

In [ ]:
sub_tg          = test_tg[['id']].copy()
sub_tg['target'] = pred_tg

sub_egc          = test_egc[['id']].copy()
sub_egc['target'] = pred_egc

submission = (
    pd.concat([sub_tg, sub_egc], axis=0)
    .sort_values('id')
    .reset_index(drop=True)
)

assert submission.shape[0] == len(test), 'Row count mismatch!'
assert submission['target'].isna().sum() == 0, 'NaN in predictions!'

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Submission shape:', submission.shape)
print(submission.head(10))
print('\nsubmission.csv saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_tg, oof_tg, alpha=0.25, s=8)
lo, hi = min(y_tg.min(), oof_tg.min()), max(y_tg.max(), oof_tg.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('True Tg (°C)', fontsize=12)
axes[0].set_ylabel('Pred Tg (°C)', fontsize=12)
axes[0].set_title(f'Tg OOF  R² = {r2_tg:.4f}', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_egc, oof_egc, alpha=0.25, s=8, color='darkorange')
lo, hi = min(y_egc.min(), oof_egc.min()), max(y_egc.max(), oof_egc.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[1].set_xlabel('True Egc (eV)', fontsize=12)
axes[1].set_ylabel('Pred Egc (eV)', fontsize=12)
axes[1].set_title(f'Egc OOF  R² = {r2_egc:.4f}', fontsize=13)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'v2 OOF Predicted vs Actual  |  Mean R² = {(r2_tg+r2_egc)/2:.4f}   (v1: 0.8997)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/kaggle/working/oof_scatter_v2.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')